# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:
Заранее прощу прощения за неправильную орфографику!!!
1. в этом коде мы инициализируем переменную s как пандас серия с значаниями и индексами в виде букв, потом мы выводим элемент с инбексом б и элемент с позиции 0 это у нас 20 и 10  
2. тут у нас серия с индексами в виде називаниями городов выводом этого кода будет
Шымкент : 3 , Алматы : null , Астана : null так как мы тут сложим элементы с совподающим индексом а те элементы у которых нет совпадение сложений не будет и код не знает с чем сложить уже сущес число изза этого будет null
3. тут у нас список дейта фрейм с 3 стоками и 2 столбцами, мы тут проверяем стороки столбца х где больше 1 и тут у нас проходит только 2 и 3 а 1 стороку мы выбрасиваем и так у нас получится выводит 2 сторки и 2 столбца  
4. в этом коде у нас ответ будет Series DataFrame так как мы метод __name__ выводит тип данных этих элементов в первом у нас мы обращаемся к столбцу и достаем его как одномерный массив с элементами а во втором мы пишем двойные скобки это у нас в пандас передает список столбцов
5. так тут мы как в прошлый задача выбрасиваем 1 строку и получаем индексы отсавщихся элементов и преврящаем их в лист это у нас [1, 2] так как мы уже стороку с индексом 0 выбросили а на счет метода iloc тут этот метод считает сам позиций с нуля мы тут ищем элемент с индексом строки 0 и с индексом столбца 1 это у нас 5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [14]:
# 1
print(df.shape)
print(df.dtypes)
print(df["delivery_days"].isna().sum())
dif = df[["city", "category", "status", "customer_id"]].nunique()
print(dif)
# Первая проблема это много пропусков в  delivery_days это может мешать будущему анализу,
# и вторая проблема это стороковый формат для order_date и customer_id, думаю дата заказа должен быть в формате даты и кастомер айди в числовом формате
# 3 проблема думаю тестовые данные они идут вместе с реальными данными

(5463, 8)
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
160
city             14
category          5
status            2
customer_id    2003
dtype: int64


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [35]:
# 2
clean = df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]
real = clean["amount"]
print(real.mean(),real.median(), real.quantile(0.25), real.quantile(0.75))
# типичный заказ думаю равно медиане так как оно больше чеснее чем среднее,
#у нас среднее большое число оно может вырасти если есть заказы с огроным чеком а медиана будет примерно показывать что половина заказов дешевле а другая половна дороже

14672.391348704346 10300.0 5880.0 18690.0


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [46]:
# 3
Al = ["Алматы", "Алма-Ата", "Almaty"]
Shym = df[df['city'].isin(SH)]
Ala = df[df['city'].isin(Al)]
res_sh = (Shym['status'] == 'отменён').mean()
res_al = (Ala['status'] == 'отменён').mean()
print(res_al, res_sh)
print(max(res_al, res_sh) / min(res_al, res_sh))
# тут мы видем что у Шымкента большой процент отмены заказов примено 20 процентов а у Алматы 8 и разница между ними это 2,5 раза больше

0.07915831663326653 0.20398009950248755
2.5768625228288933


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [93]:
# 4
sep_al = clean[clean['city'].isin(Al) & sep]
oct_al = clean[clean['city'].isin(Al) & octo ]
sep_sh = clean[clean['city'].isin(SH) & sep]
oct_sh = clean[clean['city'].isin(SH) & octo]

al_change = (oct_al["amount"].sum() - sep_al["amount"].sum()) / sep_al["amount"].sum() * 100
sh_change = (oct_sh["amount"].sum() - sep_sh["amount"].sum()) / sep_sh["amount"].sum() * 100

print("Алматы:", round(al_change, 2), "%")
print("Шымкент:", round(sh_change, 2), "%")
# здесь мы видем что выручка в обеих городах в октябре меньше чем в сеньтябре

Алматы: -1.99 %
Шымкент: -18.3 %


/tmp/ipykernel_2334/2337639284.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sep_al = clean[clean['city'].isin(Al) & sep]
/tmp/ipykernel_2334/2337639284.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  oct_al = clean[clean['city'].isin(Al) & octo ]
/tmp/ipykernel_2334/2337639284.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sep_sh = clean[clean['city'].isin(SH) & sep]
/tmp/ipykernel_2334/2337639284.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  oct_sh = clean[clean['city'].isin(SH) & octo]


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [100]:
# 5
cancel_sh = df[(df["city"].isin(["Шымкент", "шымкент", "Shymkent"]) ) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [101]:
# 6
print(df["customer_id"].value_counts().head(5))

top_id = df["customer_id"].value_counts().index[0]
mask = df["customer_id"] == top_id
print(df.loc[mask, ["order_date", "city", "amount", "status"]])

print(len(df[mask]))
print(df.loc[mask, "order_id"].nunique())

customer_id
C01818    11
C01076     9
C01058     9
C00420     9
C01947     9
Name: count, dtype: int64
            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен
11
10


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [102]:
# 7
clean = df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")]

q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1
border = q3 + 1.5 * iqr

high = clean[clean["amount"] > border]
print(len(high))
print(len(high) / len(clean) * 100)
print(high["category"].value_counts().head(1))

320
6.529279738828811
category
Электроника    308
Name: count, dtype: int64


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [103]:
# 8
cancel = df[df["status"] == "отменён"]["city"].value_counts()
total = df["city"].value_counts()

rate = (cancel / total).sort_values(ascending=False)
print(rate)

SH = ["Шымкент", "шымкент", "Shymkent"]
print(total.loc[SH])

city
Shymkent     0.230769
Шымкент      0.215190
алматы       0.140625
астана       0.130952
Актобе       0.103275
Astana       0.100000
Караганда    0.089219
Астана       0.084829
Алматы       0.079917
Атырау       0.075342
Almaty       0.057971
 Алматы      0.045455
шымкент      0.037037
АЛМАТЫ       0.032258
Name: count, dtype: float64
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64
